In [ ]:
import random
import sys
from importlib import reload
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.model_selection import KFold

sys.path.append("..")
from src import data_functions as datafun
from src import plotly_plots as pp
from src import util


In [ ]:
reload(util)
reload(datafun)

split_idx, split_date = util.load_split_idx()
print(f"Loaded split {split_date}")
data_tr = util.dataset_to_df(
    util.load_dataset_splits(split_idx, path=Path("../data/dataset.ndjson"))["train"]
)
# get a vocab
vocab, token2idx, tag_vocab, tag2idx = util.make_vocab(data_tr)

# groups etc
data_tr, group_counts = datafun.make_example_groups(data_tr, min_group_count=4)

display(data_tr.tail())

print("groups:", group_counts)
N_SAMPLES = len(data_tr)
print(f"{N_SAMPLES=}")

In [ ]:
reload(datafun)


folds = datafun.simple_folds(data_tr, 4, shuffle=True)
for s_train, s_test in folds:
    print(f"{s_train.shape}, {s_test.shape}, sum  {len(s_train) + len(s_test)}")

overlaps = [
    datafun.overlap_split_pair(
        s_train["tokens"].to_list(),
        s_test["tokens"].to_list(),
        n=3,
        norm="iou",
    )
    for s_train, s_test in folds
]
for ovr in overlaps:
    print(f"{ovr = :.2%}")